In [84]:
import re
from typing import List
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd
import glob
import os

In [48]:
train_df = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/training_data/dataset_reports_subset_from_full_data_1__sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__2nd_approach__nace_level_1__cos_thres_0.4/full_data.csv")

#test_reports = glob.glob("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_3/TXTs/*.txt")
test_reports = glob.glob("projects/nace_classification/nace_report_topic_analysis/results/BERT_classification/model_1_0__dataset__reports_subset_from_full_data_3_sentence_len_6__nace_level_1/*/*.csv")

In [39]:
# --------------------------
# 1. Simple sentence splitter
# --------------------------
def split_into_sentences(text: str) -> List[str]:
    """
    Very simple sentence splitter.
    Splits on '.', '!' or '?' followed by whitespace.
    Not perfect, but good enough to illustrate the idea.
    """
    text = text.strip()
    if not text:
        return []

    # Split on end-of-sentence punctuation
    parts = re.split(r'(?<=[.!?])\s+', text)
    # Remove empty pieces
    sentences = [p.strip() for p in parts if p.strip()]
    return sentences

In [40]:
# -------------------------------------------
# 2. Build TF-IDF model on training sentences
# -------------------------------------------
def build_tfidf_model(training_texts: List[str]) -> TfidfVectorizer:
    """
    training_texts: list of full report texts (strings)
    Returns a fitted TfidfVectorizer over all sentences in the training corpus.
    """

    # Initialize the TF-IDF vectorizer.
    # You can remove 'stop_words' if you don't want English stopword removal.
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english"  # optional
    )

    vectorizer.fit(training_texts)
    return vectorizer

In [65]:
# ------------------------------------------------
# 3. Rank sentences in a single report using TF-IDF
# ------------------------------------------------
def rank_sentences(report_paragraphs: list, vectorizer: TfidfVectorizer) -> List[tuple]:
    """
    report_text: the text of a single report
    vectorizer: a fitted TfidfVectorizer
    Returns a list of (sentence, score, original_index) sorted by score (desc).
    """

    # Transform sentences into TF-IDF matrix: shape (num_sentences, vocab_size)
    tfidf_matrix = vectorizer.transform(report_paragraphs)

    # Simple sentence score: sum of TF-IDF values of all terms in that sentence
    # shape: (num_sentences, 1) -> flatten to (num_sentences,)
    scores = np.asarray(tfidf_matrix.sum(axis=1)).ravel()

    # Create list of (sentence, score, original_index)
    sentence_info = [
        (report_paragraphs[i], float(scores[i]), i) for i in range(len(report_paragraphs))
    ]

    # Sort by score descending
    sentence_info_sorted = sorted(sentence_info, key=lambda x: x[1], reverse=True)
    return sentence_info_sorted


In [81]:
# ----------------------------------------------------------------------
# 4. Select top sentences under a word budget and restore original order
# ----------------------------------------------------------------------
def select_top_sentences(
    report_paragraphs: list,
    vectorizer: TfidfVectorizer,
    max_words: int = 512
) -> str:
    """
    report_text: text of a single report
    vectorizer: fitted TfidfVectorizer
    max_words: approximate word budget (e.g. ~512 words ~= 512-800 tokens)

    Returns a shortened text containing only the selected sentences
    in their ORIGINAL order.
    """
    # Rank sentences by importance
    ranked = rank_sentences(report_paragraphs, vectorizer)
    if not ranked:
        return ""

    # Greedy selection: go down ranked list, keep sentence if it fits in budget
    selected_indices = []
    word_count = 0

    for sentence, score, idx in ranked:
        num_words = len(sentence.split())
        if word_count + num_words <= max_words:
            selected_indices.append(idx)
            word_count += num_words

    # Restore original sentence order
    selected_indices_sorted = sorted(selected_indices)

    # Build final shortened text
    selected_sentences = [report_paragraphs[i] for i in selected_indices_sorted]
    shortened_text = " ".join(selected_sentences)

    return selected_sentences, ranked


In [67]:
training_reports = train_df["text"].to_list()

In [73]:
#Build TF-IDF model on all sentences from the training corpus
tfidf_vectorizer = build_tfidf_model(training_reports)

In [104]:
# Example new report that we want to shorten
report = test_reports[3]
new_report = pd.read_csv(report)
print(os.path.basename(report))

shortened, ranked = select_top_sentences(
    report_paragraphs=new_report["Sentences"].to_list(),
    vectorizer=tfidf_vectorizer,
    max_words=2000
)

print("=== ORIGINAL REPORT ===")
print(new_report["Sentences"].head())
print("\n=== SHORTENED REPORT ===")
for s in shortened: 
    print(s) 

Whitbread PLC3.txt_classifications.csv
=== ORIGINAL REPORT ===
0    ireland employment wage subsidy scheme jersey ...
1    m adjusted operating profitloss depreciation r...
2    we use a range of measures to monitor the fina...
3    an impairment loss of nil .m was recognised re...
4    consolidated income statement m m current tax ...
Name: Sentences, dtype: object

=== SHORTENED REPORT ===
we also continued to fundraise for great ormond street hospital gosh bringing the total raised to million since our partnership began in . a highlight this year was the opening of the sight and sound centre supported by premier inn which is the uks first dedicated medial facility for children with sight and hearing loss. having met our fundraising target of m a companywide vote was held to decide the future direction of our charity partnership. with over employee votes cast gosh was selected for a renewed term demonstrating the commitment and engagement this partnership represents for our teams. wh

In [105]:
ranked

[('the groups flexible balance sheet has enabled a programme of investment in expansion and commercial initiatives that are driving market share gains. the rest easy multichannel marketing campaign launched in april helping further improve our already high brand recognition scores and driving increased numbers of customers to the premier inn website. our targeted refurbishment capex is ahead of precovid levels albeit disrupted in the second half of fy by shortterm supply chain issues. these high levels of spend will ensure that our hotel estate remains wellinvested at a time when others will be constrained. we now have over premier plus rooms and will rollout a further in fy. these upgraded rooms were initially targeted at business customers but they have also proved popular with our leisure guests and are delivering a good arr',
  7.762258870617343,
  443),
 ('the hotel market in germany is recovering at a slower pace than the uk due to the higher level of government restrictions whic

In [ ]:
k = 10
new_report = new_report.iloc[np.array(ranked)[:,2][:k]]

,Unnamed: 0,F,P,L,K,I,A,D,G,E,NO_CLASS,B,J,H,Q,M,N,C,Sentences
443,443,2.252235,-1.935217,-1.794911,-0.782000,6.458341,-3.096761,-3.027655,-0.861194,0.080912,3.637581,-1.726503,-4.160595,-0.989216,-2.012115,-2.704738,0.199476,-0.995195,the groups flexible balance sheet has enabled ...
255,255,1.642575,-1.739769,-1.215765,-0.889471,7.736259,-2.980172,-3.060805,-0.724230,0.187416,2.307799,-1.642782,-3.954051,-1.147983,-2.359525,-2.940269,0.453505,-1.443556,the hotel market in germany is recovering at a...
479,479,1.374347,-0.882068,-4.375029,0.090319,0.127568,-3.302070,-3.501017,-1.073988,-1.927244,9.212964,-3.591138,-3.059980,-0.418258,0.747556,0.145738,0.127287,-0.179889,following on from an ambitious move to bring o...
389,389,1.321243,-1.028189,-4.448510,0.093677,0.340566,-3.309862,-3.537649,-0.920148,-1.824367,9.230917,-3.537518,-3.217304,-0.319640,0.560680,-0.071272,0.204930,-0.152644,our enlarged estate now provides us with the f...
505,505,1.388029,-0.917398,-4.396007,0.059154,0.286849,-3.340971,-3.481951,-1.047148,-1.892598,9.197918,-3.596248,-3.129987,-0.343617,0.730474,0.009239,0.143287,-0.210379,enhanced structural opportunities prior to the...
254,254,1.373394,-0.890706,-4.397094,0.066811,0.393434,-3.332444,-3.510842,-1.069653,-1.877452,9.205848,-3.599355,-3.144984,-0.374333,0.715648,-0.037714,0.175263,-0.263948,whitbreads performance in the year was strong ...
390,390,1.659180,-1.704600,-1.751509,-0.906291,7.251534,-2.948117,-3.165777,-0.800256,0.191537,3.070365,-1.858405,-4.074944,-1.170360,-2.170378,-2.776643,0.500188,-1.213547,we put our customers at the heart of our busin...
217,217,1.603671,-1.506683,-3.975463,0.846105,1.792935,-3.159246,-3.518512,-0.833613,-1.194504,8.439768,-2.933572,-3.717020,-0.409346,-0.591923,-1.389562,0.481086,-0.438261,financial flexibility the groups strong balanc...
152,152,1.371367,-0.883934,-4.371797,0.090228,0.108094,-3.328033,-3.487724,-1.068198,-1.955151,9.199146,-3.590546,-3.055611,-0.357926,0.795353,0.121182,0.120013,-0.210694,we also continued to fundraise for great ormon...
381,381,2.105618,-1.561437,-3.929293,0.033217,2.187233,-3.246337,-3.542372,-0.678389,-1.161719,8.094126,-3.069109,-3.849945,-0.427003,-0.582464,-1.213360,0.317712,-0.279571,we hold a uniquely advantaged position in the ...
